In [4]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Get the token from environment
hf_token = os.getenv("HF_Token")

# Login to Hugging Face
login(hf_token)

In [2]:
from datasets import load_dataset
from tqdm import tqdm
import time
import os
import json

def save_dataset_until_tokens(ds, filename, lang_name, max_tokens, save_every=5000, buffer_size=1000):
    """
    Save dataset to file until reaching max_tokens.
    Supports resuming with index caching (.meta file).
    Counts tokens by splitting on whitespace.
    
    Args:
        ds: HuggingFace dataset
        filename: Output file path
        lang_name: Language name for display
        max_tokens: Maximum tokens to save
        save_every: Save progress every N lines
        buffer_size: Number of lines to batch in memory before writing
    """

    meta_file = filename + ".meta"
    start_line, total_tokens, total_bytes = -1, 0, 0

    # --- Resume from .meta if exists ---
    if os.path.exists(meta_file):
        with open(meta_file, "r", encoding="utf-8") as f:
            meta = json.load(f)
            start_line = meta["last_index"]
            total_tokens = meta["tokens"]
            total_bytes = meta["bytes"]

        print(f"🔄 Resuming from {filename}")
        print(f"   Already written: {start_line+1} lines, {total_tokens:,} tokens, {total_bytes/1024/1024:.2f} MB")

        f_out = open(filename, "a", encoding="utf-8")

        # --- Prevent duplicate line on crash ---
        with open(filename, "rb") as f_check:
            f_check.seek(-200, os.SEEK_END)  # read last ~200 bytes
            last_line = f_check.read().decode("utf-8", errors="ignore").splitlines()[-1].strip()
    else:
        print(f"🆕 Starting fresh for {lang_name}")
        f_out = open(filename, "w", encoding="utf-8")
        last_line = None

    start_time = time.time()
    buffer = []

    with f_out, tqdm(unit=" lines", desc=f"Downloading {lang_name}", initial=start_line+1) as pbar:
        for i, ex in enumerate(ds):
            if i <= start_line:
                continue  # skip already written

            text = ex["text"].strip()

            # --- Skip duplicate if crash happened after writing but before saving meta ---
            if last_line and i == start_line + 1 and text == last_line:
                print("⚠️ Duplicate last line detected — skipping to avoid double write.")
                last_line = None
                continue

            tokens = text.split()
            token_count = len(tokens)

            if total_tokens + token_count > max_tokens:
                break

            buffer.append(text + "\n")
            total_tokens += token_count
            total_bytes += len(text.encode("utf-8"))

            # --- Write buffer periodically ---
            if len(buffer) >= buffer_size:
                f_out.writelines(buffer)
                buffer = []

            # --- Save progress periodically ---
            if i % save_every == 0:
                f_out.flush()
                os.fsync(f_out.fileno())

                tmp_file = meta_file + ".tmp"
                with open(tmp_file, "w", encoding="utf-8") as f:
                    json.dump({
                        "last_index": i,
                        "tokens": total_tokens,
                        "bytes": total_bytes
                    }, f)
                os.replace(tmp_file, meta_file)

            # --- Update progress bar ---
            elapsed = time.time() - start_time
            if elapsed > 0:
                speed = total_bytes / (elapsed * 1024 * 1024)  # MB/s
                pbar.set_postfix({
                    "Tokens": f"{total_tokens/1e6:.2f}M",
                    "Size": f"{total_bytes/1024/1024:.2f} MB",
                    "Speed": f"{speed:.2f} MB/s"
                })

            pbar.update(1)

        # --- Write any leftover buffer ---
        if buffer:
            f_out.writelines(buffer)

    # --- Final save ---
    tmp_file = meta_file + ".tmp"
    with open(tmp_file, "w", encoding="utf-8") as f:
        json.dump({
            "last_index": i,
            "tokens": total_tokens,
            "bytes": total_bytes
        }, f)
    os.replace(tmp_file, meta_file)

    print(f"\n✅ Finished saving {lang_name} → {filename}")
    print(f"   Total lines: {i+1}, Tokens: {total_tokens:,}, Size: {total_bytes/1024/1024:.2f} MB")
    print(f"   Progress saved in {meta_file}")

In [6]:
# Load streaming datasets
mni_ds = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split="mni_Mtei", streaming=True)

# Save Manipuri (limit for test)
save_dataset_until_tokens(mni_ds, "data/ai4bharat:IndicCorpV2/manipuri_corpus.txt", "Manipuri", max_tokens=int(9e8))

🔄 Resuming from data/ai4bharat:IndicCorpV2/manipuri_corpus.txt
   Already written: 85914 lines, 614,127 tokens, 8.81 MB



✅ Finished saving Manipuri → data/ai4bharat:IndicCorpV2/manipuri_corpus.txt
   Total lines: 85914, Tokens: 614,127, Size: 8.81 MB
   Progress saved in data/ai4bharat:IndicCorpV2/manipuri_corpus.txt.meta


In [4]:
# Load English (Verified subset)
eng_ds = load_dataset("ai4bharat/sangraha", data_dir="verified/eng", split="train", streaming=True)

# Save until 2B tokens
save_dataset_until_tokens(eng_ds, "data/ai4bharat:sangraha/english_corpus.txt", "English", max_tokens=int(2e9))

🔄 Resuming from data/ai4bharat:sangraha/english_corpus.txt
   Already written: 4425366 lines, 1,999,999,868 tokens, 11469.36 MB



✅ Finished saving English → data/ai4bharat:sangraha/english_corpus.txt
   Total lines: 4425367, Tokens: 1,999,999,868, Size: 11469.36 MB
   Progress saved in data/ai4bharat:sangraha/english_corpus.txt.meta


In [12]:
# Load dataset (Hindi, verified)
hin_ds = load_dataset("ai4bharat/sangraha", data_dir="verified/hin", split="train", streaming=True)

# Save until 1.2B tokens
save_dataset_until_tokens(hin_ds, "data/ai4bharat:sangraha/hindi_corpus.txt", "Hindi", max_tokens=int(1.2e9))

🔄 Resuming from data/ai4bharat:sangraha/hindi_corpus.txt
   Already written: 2804537 lines, 1,199,999,691 tokens, 14866.95 MB



✅ Finished saving Hindi → data/ai4bharat:sangraha/hindi_corpus.txt
   Total lines: 2804540, Tokens: 1,199,999,982, Size: 14866.95 MB
   Progress saved in data/ai4bharat:sangraha/hindi_corpus.txt.meta


In [10]:
# Load dataset (Manipuri, verified)
mni_ds = load_dataset("ai4bharat/sangraha", data_dir="verified/mni", split="train", streaming=True)

# Save until 900M tokens
save_dataset_until_tokens(mni_ds, "data/ai4bharat:sangraha/manipuri_corpus.txt", "Manipuri", max_tokens=int(9e8))

🔄 Resuming from data/ai4bharat:sangraha/manipuri_corpus.txt
   Already written: 1128 lines, 770,513 tokens, 11.02 MB



✅ Finished saving Manipuri → data/ai4bharat:sangraha/manipuri_corpus.txt
   Total lines: 1128, Tokens: 770,513, Size: 11.02 MB
   Progress saved in data/ai4bharat:sangraha/manipuri_corpus.txt.meta


In [9]:
# Load dataset (Manipuri, verified)
mni_ds = load_dataset("HuggingFaceFW/fineweb-2", name="mni_Beng", split="train", streaming=True)

# Save until 900M tokens
save_dataset_until_tokens(mni_ds, "data/fineweb2/manipuri_corpus.txt", "Manipuri", max_tokens=int(9e8))

🆕 Starting fresh for Manipuri



✅ Finished saving Manipuri → data/fineweb2/manipuri_corpus.txt
   Total lines: 3405, Tokens: 2,318,747, Size: 40.38 MB
   Progress saved in data/fineweb2/manipuri_corpus.txt.meta


In [14]:
import requests
import zipfile
import io
import os

def download_and_extract(url, output_dir="."):
    """
    Downloads a .zip from the URL and extracts its contents into output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)
    file_name = url.split("/")[-1]   # e.g. en-mni.txt.zip
    print(f"\nDownloading: {file_name}\nFrom: {url}")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = response.headers.get('content-length')
    if total_size:
        total_size = int(total_size)
        print(f"File size: {total_size / (1024*1024):.2f} MB")
    else:
        print("Size unknown — streaming...")

    bytes_io = io.BytesIO()
    downloaded = 0
    chunk_size = 8192

    for chunk in response.iter_content(chunk_size=chunk_size):
        if chunk:
            bytes_io.write(chunk)
            downloaded += len(chunk)
            if total_size:
                percent = downloaded / total_size * 100
                print(f"\rDownloaded: {downloaded / (1024*1024):.2f} MB ({percent:.1f}%)", end="")
    print("\nDownload complete.")

    bytes_io.seek(0)

    print("Extracting contents...")
    with zipfile.ZipFile(bytes_io) as z:
        z.extractall(output_dir)
    print(f"✅ Extracted into: {output_dir}")


if __name__ == "__main__":
    BASE_OUTPUT_DIR = "data/opus/"

    urls = [
        "https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/en-mni.txt.zip",
        "https://object.pouta.csc.fi/OPUS-wikimedia/v20230407/moses/en-mni.txt.zip",
        "https://object.pouta.csc.fi/OPUS-pmindia/v1/moses/en-mni.txt.zip",
        "https://object.pouta.csc.fi/OPUS-NLLB/v1/moses/en-mni.txt.zip",
    ]

    for url in urls:
        dataset_name = url.split("/")[-3]   # e.g. OPUS-Tatoeba
        out_dir = os.path.join(BASE_OUTPUT_DIR, dataset_name)
        download_and_extract(url, output_dir=out_dir)


Downloading: en-mni.txt.zip
From: https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/en-mni.txt.zip
File size: 0.01 MB
Downloaded: 0.01 MB (100.0%)
Download complete.
Extracting contents...
✅ Extracted into: data/opus/v2023-04-12

Downloading: en-mni.txt.zip
From: https://object.pouta.csc.fi/OPUS-wikimedia/v20230407/moses/en-mni.txt.zip
File size: 0.22 MB
Downloaded: 0.22 MB (100.0%)
Download complete.
Extracting contents...
✅ Extracted into: data/opus/v20230407

Downloading: en-mni.txt.zip
From: https://object.pouta.csc.fi/OPUS-pmindia/v1/moses/en-mni.txt.zip
File size: 0.71 MB
Downloaded: 0.71 MB (100.0%)
Download complete.
Extracting contents...
✅ Extracted into: data/opus/v1

Downloading: en-mni.txt.zip
From: https://object.pouta.csc.fi/OPUS-NLLB/v1/moses/en-mni.txt.zip
File size: 23.90 MB
Downloaded: 23.90 MB (100.0%)
Download complete.
Extracting contents...
✅ Extracted into: data/opus/v1


In [ ]:
import os

# input_dir = "data/ai4bharat:IndicCorpV2/"
# input_dir = "data/ai4bharat_sangraha_mni/"
# input_dir = "data/fineweb2/"
# input_dir = "data/kaggle/"
# input_dir = "data/manipuri/"
# input_dir = "data/opus/"
# input_dir = "data/pmindia.v1.mni/"
output_file = "data/manipuri_corpus.txt"

# Open output file in append mode (so new runs just keep adding text)
with open(output_file, "a", encoding="utf-8") as outfile:
    # Walk through the directory
    for root, _, files in os.walk(input_dir):
        for filename in files:
            if filename.endswith(".txt"):
                file_path = os.path.join(root, filename)
                try:
                    with open(file_path, "r", encoding="utf-8") as infile:
                        text = infile.read()
                        outfile.write(text + "\n")  # add a newline between files
                    print(f"Added: {file_path}")
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

print("✅ All files combined into:", output_file)

Added: data/kaggle/1_manipuri (bengali Scirpt).txt
✅ All files combined into: data/manipuri_corpus.txt


In [3]:
# Load streaming datasets
mni_ds = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split="ben_Beng", streaming=True)

# Save Bengali (limit for test)
save_dataset_until_tokens(mni_ds, "data/ai4bharat:IndicCorpV2/bengali_corpus.txt", "Bengali", max_tokens=int(9e8))

🔄 Resuming from data/ai4bharat:IndicCorpV2/bengali_corpus.txt
   Already written: 39821671 lines, 899,999,992 tokens, 14773.58 MB



✅ Finished saving Bengali → data/ai4bharat:IndicCorpV2/bengali_corpus.txt
   Total lines: 39821673, Tokens: 899,999,992, Size: 14773.58 MB
   Progress saved in data/ai4bharat:IndicCorpV2/bengali_corpus.txt.meta


In [4]:
import os

file_paths = [
    "data/english_corpus.txt",
    "data/hindi_corpus.txt",
    "data/bengali_corpus.txt",
    "data/manipuri_corpus.txt"
]

print("📊 Corpus Statistics")

for file_path in file_paths:
    num_lines = 0
    num_tokens = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            num_lines += 1
            tokens = line.strip().split()
            num_tokens += len(tokens)

    # Get file size in MB
    file_size = os.path.getsize(file_path) / (1024 * 1024)

    print(f"\nFile: {file_path}")
    print(f"  Total lines   : {num_lines}")
    print(f"  Total tokens  : {num_tokens}")
    print(f"  File size     : {file_size:.2f} MB")

📊 Corpus Statistics

File: data/english_corpus.txt
  Total lines   : 48529117
  Total tokens  : 1802671036
  File size     : 10342.11 MB

File: data/hindi_corpus.txt
  Total lines   : 22643156
  Total tokens  : 1201257656
  File size     : 14885.12 MB

File: data/bengali_corpus.txt
  Total lines   : 39821670
  Total tokens  : 899999992
  File size     : 14811.56 MB

File: data/manipuri_corpus.txt
  Total lines   : 785908
  Total tokens  : 9128751
  File size     : 156.74 MB


# Translation

In [ ]:
"""Run MT API."""
# Language codes for passing onto the API, 3 lettered language identifier.
# Kannada - kan, Punjabi - pun, Urdu - urd, Telugu - tel, Hindi - hin, Oriya - oryy,
# Bengali - ben, Tamil - tam, Malayalam - mal, Assamese - asm, Gujarati - guj,
# Marathi - mar, Nepali - nep, Maithili - mai, English - eng
# For example, to translate from Hindi to English, use hin and eng respectively.
# English - eng
# if requests package is not there, do pip install requests
import requests
import json
from sys import argv


url = "https://ssmt.iiit.ac.in/onemt"


def run_mt_api_with_arguments(text, source, target):
    """Run MT API with arguments."""
    out =  {"source_language": source, "target_language": target, "text": text, "mode": "versionvMD"}
    headers = {'Content-type': 'application/json', 'Accept': 'text/plain'}
    r = requests.post(url, data=json.dumps(out), headers=headers)
    json_output = r.json()
    target_text = json_output['data']
    return target_text


def main():
    """Pass arguments and call functions here."""
    text = argv[1]
    src_lang = argv[2]
    tgt_lang = argv[3]
    target_text = run_mt_api_with_arguments(text, src_lang, tgt_lang)
    print(target_text)


# This script can be run from the command line with three arguments:
# 1. The text to be translated
# 2. The source language code
# 3. The target language code
# Example usage:
# python run_mt_api_with_arguments.py "Hello, how are you?" eng hin
# This will translate "Hello, how are you?" from English to Hindi.
# Make sure to replace the text and language codes as needed.
# The output will be printed to the console.
# Ensure you have the requests library installed in your Python environment.
# You can install it using pip:
# pip install requests
# The API endpoint is set to the OneMT service at IIIT Hyderabad.
# The service supports various Indian languages, and you can specify the source and target languages using their
# respective language codes.
# The response from the API will be in JSON format, and the translated text will be extracted from the response.
# The script is designed to be run from the command line, making it easy to use for quick translations.
# Make sure to handle any exceptions or errors that may arise during the API call, such as network issues or invalid inputs.
# You can enhance the script by adding error handling to manage cases where the API might not return a valid response or if the input text is empty.
# This script is a simple example of how to interact with a machine translation API using Python and the requests library.
# It can be extended or modified to suit more complex use cases, such as batch translations or integrating with other applications.
# Always check the API documentation for any updates or changes in the request format or available languages.
# The script assumes that the API is available and functioning correctly.
# If the API is down or the endpoint changes, the script will not work as intended.
if __name__ == '__main__':
    main()

In [ ]:
#!/usr/bin/env python3
"""
Convert Bengali corpus -> Manipuri (Bengali script) using IndicTrans2 model.
Resumable via OUTPUT_FILE.meta (stores next_index). Simple, robust.
"""

import os, json, time
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IndicTransToolkit.processor import IndicProcessor

# --------- CONFIG ----------
INPUT_FILE  = "data/bengali_corpus.txt"
OUTPUT_FILE = "data/bengali_to_mni.txt"
META_FILE   = OUTPUT_FILE + ".meta"

MODEL_NAME  = "ai4bharat/indictrans2-indic-indic-1B"
BATCH_SIZE  = 64       # adjust to memory; larger is faster but uses more RAM/VRAM
SAVE_EVERY  = 500      # save meta every N batches
BUFFER_SIZE = 200      # write to disk every this many translated lines
# --------------------------

# device + dtype selection
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
TORCH_DTYPE = (torch.float16 if DEVICE == "cuda" else torch.float32)
USE_CACHE = True if DEVICE == "cuda" else False   # disable cache on cpu/mps to avoid KeyError

print(f"Device={DEVICE}  dtype={TORCH_DTYPE}  use_cache={USE_CACHE}")

# load model/tokenizer/processor
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=TORCH_DTYPE).to(DEVICE)
ip = IndicProcessor(inference=True)

# consistent language tags (Bengali script for Manipuri)
SRC_LANG = "ben_Beng"
TGT_LANG = "mni_Beng"

# read input lines (strip empty)
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    all_lines = [ln.rstrip("\n") for ln in f if ln.strip()]

total_lines = len(all_lines)
print(f"Total lines in corpus: {total_lines}")

# load resume meta if exists
next_index = 0
if os.path.exists(META_FILE):
    with open(META_FILE, "r", encoding="utf-8") as mf:
        meta = json.load(mf)
        next_index = meta.get("next_index", 0)
    print(f"🔁 Resuming from index {next_index}")

# open output file in append or write mode based on resume
out_mode = "a" if next_index > 0 else "w"
out_file = open(OUTPUT_FILE, out_mode, encoding="utf-8")

buffer = []
batches_written = 0
start_time = time.time()

try:
    # iterate in batches
    with tqdm(total=total_lines, desc="Translating", unit="lines") as pbar:
        pbar.update(next_index)  # show progress already done
        i = next_index
        while i < total_lines:
            batch_lines = all_lines[i: i + BATCH_SIZE]

            # preprocess -> tokenization expects language-tagged strings from IndicProcessor
            batch_for_tokenizer = ip.preprocess_batch(batch_lines, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

            # tokenize and move to device
            inputs = tokenizer(batch_for_tokenizer, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

            # generate (disable cache on non-cuda)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    use_cache=USE_CACHE,      # critical fix
                    max_length=256,
                    num_beams=5,
                    num_return_sequences=1,
                    early_stopping=True,
                )

            # decode & postprocess
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            translated = ip.postprocess_batch(decoded, lang=TGT_LANG)

            # add to buffer (each as a newline)
            buffer.extend(line + "\n" for line in translated)

            i += len(batch_lines)
            pbar.update(len(batch_lines))

            # flush buffer periodically
            if len(buffer) >= BUFFER_SIZE:
                out_file.writelines(buffer)
                out_file.flush()
                os.fsync(out_file.fileno())
                buffer = []

            # save meta periodically (store next index to process)
            if (i // BATCH_SIZE) % SAVE_EVERY == 0:
                tmp = META_FILE + ".tmp"
                with open(tmp, "w", encoding="utf-8") as tm:
                    json.dump({"next_index": i}, tm)
                os.replace(tmp, META_FILE)

    # end while
except KeyboardInterrupt:
    print("\n⚠️ Interrupted by user — saving progress...")
except Exception as e:
    print(f"\n❌ Error occurred at index {i}: {e}\nSaving progress before exit...")

# final flush + write meta
if buffer:
    out_file.writelines(buffer)
    out_file.flush()
    os.fsync(out_file.fileno())
out_file.close()

# final meta write (next_index = i)
final_next = min(i, total_lines)
with open(META_FILE + ".tmp", "w", encoding="utf-8") as tm:
    json.dump({"next_index": final_next}, tm)
os.replace(META_FILE + ".tmp", META_FILE)

elapsed = time.time() - start_time
lines_done = final_next
print(f"\n✅ Done up to index {final_next}/{total_lines} ({lines_done/total_lines*100:.2f}%)")
print(f"Elapsed {elapsed/60:.2f} min, avg {lines_done/elapsed:.2f} lines/sec")